# Low-Rank Adaptation (LoRA)

In [1]:
import torch
from datasets import load_dataset
from peft import get_peft_model, LoraConfig
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer, Trainer, TrainingArguments

In [2]:
# Loading the pre-trained model and its corresponding tokenizer
name = "distilbert-base-uncased"
pretrained_model = DistilBertForSequenceClassification.from_pretrained(name, num_labels=2)
tokenizer = DistilBertTokenizer.from_pretrained(name)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
# Loading the IMDB dataset and using only first 500 rows for training
data = load_dataset("stanfordnlp/imdb", split="train[:500]")

In [4]:
data[:4]

{'text': ['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

In [5]:
# Tokenize the input data
def tokenize_function(reviews):
    return tokenizer(reviews["text"], padding="max_length", truncation=True)


tokenized_data = data.map(tokenize_function, batched=True)

In [6]:
tokenized_data[:4]

{'text': ['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far b

In [7]:
tokenized_data.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [8]:
# Defining LoRA config
peft_config = LoraConfig(
        r=8,
        lora_alpha=32,
        target_modules=["q_lin", "v_lin"],
        lora_dropout=0.1,
        bias="none"
)
# Applying the defined config to the model
peft_model = get_peft_model(pretrained_model, peft_config)

In [9]:
# Defining training arguments
args = TrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=8,
        logging_dir="./logs",
        logging_steps=10,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [10]:
# Initializing the Trainer
trainer = Trainer(
        model=peft_model,
        args=args,
        train_dataset=tokenized_data,
)

In [11]:
# Train the model
trainer.train()
# Save the model
peft_model.save_pretrained("./peft-distilbert-imdb")

Step,Training Loss
10,0.687132
20,0.666584
30,0.641581
40,0.608360
50,0.578372
60,0.524779
70,0.451710
80,0.367983
90,0.317880
100,0.267171


In [12]:
# Sentence to run inference on
test_sentence = "I love this horrible shite movie!"
# Tokenizing the sentence
input = tokenizer(test_sentence, return_tensors="pt", padding=True, truncation=True)

# Running inference
with torch.no_grad():
    input = {k: v.to(peft_model.device) for k, v in input.items()}
    output = peft_model(**input)
    logits = output.logits
    predicted_class = torch.argmax(logits, dim=-1).item()

# Mapping the predicted label to the actual label
label_mapping = {0: "negative", 1: "positive"}
predicted_review = label_mapping[predicted_class]
print(f"Test sentence is: {test_sentence}")
print(f"Predicted sentiment is: {predicted_review}")

Test sentence is: I love this horrible shite movie!
Predicted sentiment is: negative
